# Real data can be complicated
Real world data can be difficult to work with. We will look at a few difficulties that often occur when working with real data that we read into pandas. These difficulties usually involve dealing with data that are badly formatted and/or spread across multiple files.

## Learning outcome:
Working with different type of difficulties that arise in real data
- Not having a header row for the data
- Comment lines in the data
- Footer following the data
- Inconsistent data (eg., boolean values, separators)
- Empty columns
- Missing values

In [ ]:
import pandas as pd
awkward_file = "https://raw.githubusercontent.com/csbfx/advpy122-data/master/awkward_input.csv"

## Inconsistent data
Sometimes files are just a clean like a comma- or tab-delimited file. Such as having header lines, comment lines, or footer.

In [ ]:
awkward = pd.read_csv(awkward_file)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 2


In [ ]:
import urllib.request

### Read the first few lines
with urllib.request.urlopen(awkward_file) as response:
    for i in range(15):
        line = response.readline().decode('utf-8').strip()
        print(f"{i + 1}: {line}")

1: Data file prepared on 4/10/2019
2: Original data taken from https://www.ncbi.nlm.nih.gov/
3: From left to right, the columns are:
4: species name
5: genome size in megabases
6: gc percentage
7: number of genes
8: publication year
9: assembly status
10: whether the genome belongs to an animal
11: whether the genome belongs to a plant
12: 
13: 
14: Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True,,
15: Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True,,


### Address the comment lines and lack of header
We observe that the data does not start until after line 11. We can address that by using the parameter `skiprows=`. Additionally, pandas expect a header when it creates a DataFrame from the csv file, but what if you have no header? We can pass the parameter `header=None` to tell pandas there is no header line.

In [ ]:
awkward = pd.read_csv(awkward_file
                     )

,0,1,2,3,4,5,6,7,8,9
0,Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True,NaN,NaN
1,Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True,NaN,NaN
2,Medicago truncatula,"412,924",34.047,37_603,2011.0,Chromosome,False,True,NaN,NaN
3,Solanum lycopersicum,"828,349",35.6991,31_200,2010.0,Chromosome,no,yes,NaN,NaN
4,Hordeum vulgare,"4006,12",44.3,NaN,2019.0,Scaffold,False,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3049,Homo sapiens,"2,097",45.8,NaN,2017.0,Scaffold,1,0,NaN,NaN
3050,Homo sapiens,"4,781",44.6,NaN,2017.0,Scaffold,1,0,NaN,NaN
3051,Homo sapiens,"4,799",44.6,missing,2017.0,Scaffold,True,False,NaN,NaN
3052,Homo sapiens,"2,79",34.8,NaN,2017.0,Scaffold,True,False,NaN,NaN


### Pass in a header you define
Pandas by default assumes that the first line of data is actually the column names. If the first line of your data is not the header line that contains the column names, we can fix this by explicitly giving a list of column names that we want in the `names=` argument:

In [ ]:
my_columns = ["Species","Genome size","GC%","Genes",
              "Year","Status","Is animal","Is plant"]

awkward = pd.read_csv(awkward_file,
                      names=
                     )
awkward

Species Genome size     GC%  \
Arabidopsis thaliana               119,669  36.0529      38_311  2001.0   
Glycine max                        979,046  35.1153      59_847  2010.0   
Medicago truncatula                412,924   34.047      37_603  2011.0   
Solanum lycopersicum               828,349  35.6991      31_200  2010.0   
Hordeum vulgare                    4006,12     44.3         NaN  2019.0   
...                                             ...         ...     ...   
Homo sapiens                       2,097       45.8         NaN  2017.0   
                                   4,781       44.6         NaN  2017.0   
                                   4,799       44.6     missing  2017.0   
                                   2,79        34.8         NaN  2017.0   
This is version 2.5432 of the file NaN          NaN         NaN     NaN   

                                                 Genes   Year Status  \
Arabidopsis thaliana               119,669  Chromosome  False   True   
Glycine max                        979,046  Chromosome  False   True   
Medicago truncatula                412,924  Chromosome  False   True   
Solanum lycopersicum               828,349  Chromosome     no    yes   
Hordeum vulgare                    4006,12    Scaffold  False   True   
...                                                ...    ...    ...   
Homo sapiens                       2,097      Scaffold      1      0   
                                   4,781      Scaffold      1      0   
                                   4,799      Scaffold   True  False   
                                   2,79       Scaffold   True  False   
This is version 2.5432 of the file NaN             NaN    NaN    NaN   

                                            Is animal  Is plant  
Arabidopsis thaliana               119,669        NaN       NaN  
Glycine max                        979,046        NaN       NaN  
Medicago truncatula                412,924        NaN       NaN  
Solanum lycopersicum               828,349        NaN       NaN  
Hordeum vulgare                    4006,12        NaN       NaN  
...                                               ...       ...  
Homo sapiens                       2,097          NaN       NaN  
                                   4,781          NaN       NaN  
                                   4,799          NaN       NaN  
                                   2,79           NaN       NaN  
This is version 2.5432 of the file NaN            NaN       NaN  

[3054 rows x 8 columns]

### Empty columns
If you look at the raw data in the **awkward_input.csv**, you will see that all the data line ends with two commas. This is a fairly common occurence, especially in files that have been exported from spreadsheet programs such as Excel. These two extra commas cause pandas to assume that each row has 10 fields rather than 8, so when we give a list of only 8 column names, it uses the first two as the index. In our case, this means that all of the column names are offset by two places.

To fix this, we will need to explictly tell pandas with columns we want to use with the `usecols=` argument.

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-'],
                      skiprows=11,
                      names=my_columns,
                      usecols=
                     )
awkward

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
0,Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True
1,Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True
2,Medicago truncatula,"412,924",34.047,37_603,2011.0,Chromosome,False,True
3,Solanum lycopersicum,"828,349",35.6991,31_200,2010.0,Chromosome,no,yes
4,Hordeum vulgare,"4006,12",44.3,NaN,2019.0,Scaffold,False,True
...,...,...,...,...,...,...,...,...
3049,Homo sapiens,"2,097",45.8,NaN,2017.0,Scaffold,1,0
3050,Homo sapiens,"4,781",44.6,NaN,2017.0,Scaffold,1,0
3051,Homo sapiens,"4,799",44.6,missing,2017.0,Scaffold,True,False
3052,Homo sapiens,"2,79",34.8,NaN,2017.0,Scaffold,True,False


### There might be footer following the data
Notice the last row in the dataframe - it is obviously
not real data as is corresponds to a footer in the original file. Many real life data files include a footer like this. We can tell pandas to skip it by passing `skipfooter` (on some versions, this may cause a `ParserWarning` , but it will still work. To avoid the warning we can pass in `engine='python'`.

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      engine='python', # avoid warning
                      skipfooter=1 # skip one footer line
                     )
awkward

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
0,Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True
1,Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True
2,Medicago truncatula,"412,924",34.047,37_603,2011.0,Chromosome,False,True
3,Solanum lycopersicum,"828,349",35.6991,31_200,2010.0,Chromosome,no,yes
4,Hordeum vulgare,"4006,12",44.3,NaN,2019.0,Scaffold,False,True
...,...,...,...,...,...,...,...,...
3048,Homo sapiens,"4,898",44.6,missing,2017.0,Scaffold,yes,no
3049,Homo sapiens,"2,097",45.8,NaN,2017.0,Scaffold,1,0
3050,Homo sapiens,"4,781",44.6,NaN,2017.0,Scaffold,1,0
3051,Homo sapiens,"4,799",44.6,missing,2017.0,Scaffold,True,False


### Characters in numeric columns
Numbers contains other characters other than a dot as the decimal separator. This results in pandas not recognizing the datatype properly

In [ ]:
# examine the data types
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   object 
 2   GC%          2897 non-null   object 
 3   Genes        1296 non-null   object 
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(1), object(7)
memory usage: 190.9+ KB


We notice that **Genome size**, **GC%**, and **Genes** should be numbers.

In [ ]:
awkward["Genome size"]

,Genome size
0,"119,669"
1,"979,046"
2,"412,924"
3,"828,349"
4,"4006,12"
...,...
3048,"4,898"
3049,"2,097"
3050,"4,781"
3051,"4,799"


Looking at the values, we can see that these numbers have been written with a comma rather than a dot as the decimal separator (common practice in many countries). Setting the `decimal=` argument will allow these numbers to be parsed correctly and turn our **Genome size** column into a floating point data type:

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python', # avoid warning
                      decimal=
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2897 non-null   object 
 3   Genes        1296 non-null   object 
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(2), object(6)
memory usage: 190.9+ KB


**GC%** should be numeric, let's see if there is any non-numeric data in that column.

In [ ]:
awkward["GC%"]

,GC%
0,36.0529
1,35.1153
2,34.047
3,35.6991
4,44.3
...,...
3048,44.6
3049,45.8
3050,44.6
3051,44.6


### Handling missing values
In the GC column we noticed previously that one of the value is "missing". Since this column should be numeric, we can use regular expression to find all non-numeric values in case there is more than just "-" and "missing" used to mark for null values.

Import the library for regular expression via `import re`. Using a python list comprehension and the regular expression library we will search for all non-numeric values.

In [ ]:
import re
all_GC = awkward["GC%"].dropna().unique()
non_num = [i for i in all_GC if not re.match("[.0-9]+", i)] # python list comprehension
print(non_num)

['missing']


We can add 'missing' to our list of `na_values`

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', ]
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    object 
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(3), object(5)
memory usage: 190.9+ KB


#### Inconsistent thousands separators

**Genes** should be numeric, let's see if there is any non-numeric data in that column.


In [ ]:
awkward["Genes"].head()

,Genes
0,38_311
1,59_847
2,37_603
3,31_200
4,NaN


The problem is the thousands separator, which is an underscore, thus the datatype detected is object. Setting the `thousands=` argument will allow pandas to parse these numbers properly.

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands=
                     )
awkward.info()
awkward

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    float64
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(4), object(4)
memory usage: 190.9+ KB


,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
0,Arabidopsis thaliana,119.669,36.0529,38311.0,2001.0,Chromosome,False,True
1,Glycine max,979.046,35.1153,59847.0,2010.0,Chromosome,False,True
2,Medicago truncatula,412.924,34.0470,37603.0,2011.0,Chromosome,False,True
3,Solanum lycopersicum,828.349,35.6991,31200.0,2010.0,Chromosome,no,yes
4,Hordeum vulgare,4006.120,44.3000,NaN,2019.0,Scaffold,False,True
...,...,...,...,...,...,...,...,...
3048,Homo sapiens,4.898,44.6000,NaN,2017.0,Scaffold,yes,no
3049,Homo sapiens,2.097,45.8000,NaN,2017.0,Scaffold,1,0
3050,Homo sapiens,4.781,44.6000,NaN,2017.0,Scaffold,1,0
3051,Homo sapiens,4.799,44.6000,NaN,2017.0,Scaffold,True,False


### Embedded comment lines
Now that we have all our columns in the right dtypes, take a look at the numbers of values. We have 3053 rows, but five of the columns that we expect to have no missing data have 3051 values. This is suspicious! Let’s find the two rows with a missing size and take a look at the data.

In [ ]:
### Find lines in which Year is null since we notice these are the only NaN values
awkward[awkward["Year"].isnull()]

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
28,# the next two genomes belong to frogs,NaN,NaN,NaN,NaN,None,None,None
86,# three different Caenorhabditis species,NaN,NaN,NaN,NaN,None,None,None


#### Handing the comment lines in the data
These comment lines contain no commas, so the entire line has ended up in the Species column. The remaining columns for those rows are either missing data (in the case of the numerical columns) or `None` (in the case of the string columns).

To fix this, we just have to tell pandas that lines begining with a hash symbol are comments by setting the `comment=` argument:

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands="_",
                      comment=
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3051 entries, 0 to 3050
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3051 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    float64
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(4), object(4)
memory usage: 190.8+ KB


### Inconsistent boolean values
Let’s take a look at the final two columns: **Is animal** and **Is plant**. These are examples of data types that we’ve not seen before - _boolean_ data, i.e. True/False values. We can specify what data type we want for a given column by passing a `dtype` dict, so let’s try making these boolean:

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands="_",
                      comment="#",
                      #dtype={"Is animal": bool, "Is plant": bool} ## DON'T RUN YET!!!
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3051 entries, 0 to 3050
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3051 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    float64
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(4), object(4)
memory usage: 190.8+ KB


In [ ]:
# How does python evaluate non-boolean values
x = "False"

if x:
    print("True")
else:
    print("False")

True


In [ ]:
# Let's check how many True/False in the column Is animal
awkward["Is animal"].value_counts()

,count
Is animal,
1,750
True,661
yes,446
"""t""",324
0,297
False,254
no,188
"""f""",131


This doesn't look right since we have both plants and animals data in this file. Let's try take a step back and undo setting the dtype.

The reason for the preponderance of `True` values when we tried setting the data type as boolean is because of Python’s rule for determining boolean values. The rule for strings is simple: any non-empty string counts as true. So all of the values that are in the file are interpreted as true.

In [ ]:
trueList = []
falseList = []
for val in awkward["Is animal"].unique():
    trueList.append(val)
    #print(repr(val), bool(val))


#print(trueList)
falseList = trueList[0:4]
trueList = trueList[4:]

print(falseList)
print(trueList)


['False', 'no', '"f"', '0']
['1', 'yes', '"t"', 'True']


Let’s fix the boolean values by passing `true_values` and `false_values` to explicity tell pandas what values we want to interpret as true and false. We can explicitly define our true/false values by passing in a list `true_values=['True', '1', 'yes', '"t'"]`. However, since we already created the list above, let's just pass that list in to the parameter `true_values=`.

Additionally, since we know that this column is a boolean value and we want pandas to treat it as so, let's change our datatype (`dtype=`) for those columns as boolean via `{<colname>: bool}`

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands="_",
                      comment="#",
                      # explicitly define the true/false values
                      #true_values=['True', '1', 'yes', '"t'"],
                      #false_values=['False', '0', 'no', '"f'"],
                      true_values = trueList,
                      false_values = falseList,
                      dtype={"Is animal": bool, "Is plant": bool}
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3051 entries, 0 to 3050
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3051 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    float64
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   bool   
 7   Is plant     3051 non-null   bool   
dtypes: bool(2), float64(4), object(2)
memory usage: 149.1+ KB


In [ ]:
# check to see if we have a good mixture of True and False
awkward[["Is animal","Is plant"]].apply(lambda x: x.value_counts())

,Is animal,Is plant
"""f""",131,324
"""t""",324,131
0,297,750
1,750,297
False,254,661
True,661,254
no,188,446
yes,446,188


# Let's put what we just learn to practice

blast_results: https://raw.githubusercontent.com/csbfx/advpy122-data/master/blast_results.csv

Read in the data from the csv file above. Skip all the comment lines, create a header for the dataframe based on the fields list in the comment line that starts with # Fields:. Drop the first column query acc.ver. Set the subject acc.ver as the index of the dataframe. The last column of the dataframe should be publications. Show the first five rows of the resulting dataframe.

In [ ]:
# Your code

If all of the inconsistently is dealt with properly, we should now be able to do some exploratory data analysis.  

### CL10.1
What is the average number of publications?

In [ ]:
# Your code

### CL10.2
List the `subject acc.ver` that has over 15,000 bonds.

In [ ]:
# Your code

### CL10.3
Create a plot that shows the correlation between `% identity` and `% positives`.

In [ ]:
# Your code

### CL10.4
Add a column `Protein source` to the dataframe from Question 1 based on the data in this csv file: https://raw.githubusercontent.com/csbfx/advpy122-data/master/protein_source.csv. Merge the data when the `subject acc.ver` from the dataframe equals to `Protein` in the csv file. Entries without a matching protein will have `NA` as `Protein source`.

In [ ]:
# Your code